In [3]:
#Create the Connection and database for RAG
from pymilvus import connections
from pymilvus import db, Collection
from pymilvus import utility
connections.add_connection(
    rag_conn={
        "host": "localhost",
        "port": "19530",
        "username": "username",
        "password": "password"
    })
conn_name="rag_conn"
db_name="rag_db"

connections.connect(conn_name)
connections.list_connections

current_dbs=db.list_database(using=conn_name)
print("Databases:", current_dbs)
if (db_name not in current_dbs):
    print("Creating database", db_name)
    resume_db = db.create_database(db_name, using=conn_name)
#Switch to the new database
db.using_database(db_name, using=conn_name)

D:\apps\anaconda3\envs\Milvus\Lib\site-packages\pymilvus\client\__init__.py:6: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import DistributionNotFound, get_distribution


Databases: ['default', 'course_db', 'cache_db']
Creating database rag_db


In [4]:
# Creating New Collection for RAG
from pymilvus import CollectionSchema, FieldSchema, DataType, Collection
import json

chunk_id_field = FieldSchema (
    name="chunk_id",
    dtype=DataType.INT64,
    is_primary=True,
    max_length=32)

rag_text_field = FieldSchema (
    name="rag_text",
    dtype=DataType.VARCHAR,
    max_length=2048)

rag_embedding_field = FieldSchema(
    name="rag_embedding",
    dtype=DataType.FLOAT_VECTOR,
    dim=1536 #Define based on embedding used
)

rag_schema = CollectionSchema(
    fields=[chunk_id_field, rag_text_field, rag_embedding_field],
    description="RAG Schema",
    enable_dynamic_field=True
)

collection_name="rag_collection"

rag_collection = Collection(
    name=collection_name,
    schema=rag_schema,
    using=conn_name,
    shard_num=2
)

from pymilvus import utility
print("Collections: ", utility.list_collections(using=conn_name))

r_collection = Collection(collection_name, using=conn_name)
print("Schema: ", r_collection.schema)

Collections:  ['rag_collection']
Schema:  {'auto_id': False, 'description': 'RAG Schema', 'fields': [{'name': 'chunk_id', 'description': '', 'type': <DataType.INT64: 5>, 'is_primary': True, 'auto_id': False}, {'name': 'rag_text', 'description': '', 'type': <DataType.VARCHAR: 21>, 'params': {'max_length': 2048}}, {'name': 'rag_embedding', 'description': '', 'type': <DataType.FLOAT_VECTOR: 101>, 'params': {'dim': 1536}}], 'enable_dynamic_field': True}


# Preparing data fro Knowledge Base

In [10]:
!pip install pypdf

In [17]:
#Load up the PDF document
from langchain_community.document_loaders.pdf import PyPDFLoader
loader = PyPDFLoader("llm.pdf")
pdf_docs = loader.load()

In [18]:
#Split document into chunks
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=512, # Specify the character chunk size
    chunk_overlap=32, # "Allowed" Overlap across chunks
    length_function=len # Function used to evaluate the chunk size (here in terms of characters)
)

pdf_docs = text_splitter.split_documents(pdf_docs)
#Create a list of chunks
rag_text = []
for i in pdf_docs:
    rag_text.append(i.page_content)
print("Total chunks", len(rag_text))
print("Sample chunk text: ", rag_text[1])

Total chunks 24
Sample chunk text:  engineering. These models acquire predictive power regarding syntax, semantics, and ontologies 
inherent in human language corpora, but they also inherit inaccuracies and biases present in the data 
they are trained on. They consist of billions to trillions of parameters and operate as general-purpose 
sequence models, generating, summarizing, translating, and reasoning over text. LLMs represent a 
significant new technology in their ability to generalize across tasks with minimal task-specific


In [20]:
#create embeddings
from langchain.embeddings import OpenAIEmbeddings
import os

openai_api_key="abc123"
os.environ["OΡΕΝΑΙ_ΑΡΙ_ΚΕΥ"] = openai_api_key

embeddings_model = OpenAIEmbeddings()

rag_embedding = [embeddings_model.embed_query(i) for i in rag_text]
#Create chunk IDs
record_ids = [i for i in range(len(rag_text))]

D:\apps\anaconda3\envs\Milvus\Lib\site-packages\langchain_core\_api\deprecation.py:117: LangChainDeprecationWarning: The class `langchain_community.embeddings.openai.OpenAIEmbeddings` was deprecated in langchain-community 0.1.0 and will be removed in 0.2.0. An updated version of the class exists in the langchain-openai package and should be used instead. To use it run `pip install -U langchain-openai` and import as `from langchain_openai import OpenAIEmbeddings`.
  warn_deprecated(


# Populating Milvus DB

In [21]:
insert_data = [record_ids, rag_text, rag_embedding]
i_collection = Collection(collection_name, using=conn_name)

#Insert the records
mr=i_collection.insert(insert_data)

#Flush the inserted records
i_collection.flush()

#Build an index on the embedding Field
index_params = {
    "metric_type":"L2",
    "index_type":"IVF_FLAT",
    "params":{"nlist":1024}
}

i_collection.create_index(
    field_name="rag_embedding",
    index_params=index_params
)

utility.index_building_progress(collection_name, using=conn_name)

{'total_rows': 24, 'indexed_rows': 24, 'pending_index_rows': 0}

# Answering Questions with RAG

In [29]:
#The retrieval process
search_params = {
    "metric_type": "L2",
    "offset": 0,
    "ignore_growing": False,
    "params": {"nprobe": 20, "radius":0.5}
}
query = "What is GPT-1?"
search_embed=embeddings_model.embed_query(query)
#print(search_embed)

q_collection = Collection(collection_name, using=conn_name)
q_collection.load()

results = q_collection.search(
    data=[search_embed],
    anns_field="rag_embedding",
    param=search_params,
    limit=3, #Get top 3 results only
    expr=None,
    output_fields=["rag_text"],
    consistency_level="Strong"
)
# print(results)
print("Top result:", results[0][0])

Top result: id: 11, distance: 0.4243744909763336, entity: {'rag_text': 'offering of downloading the model to execute locally. But it was the 2022 consumer-facing chatbot \nChatGPT that received extensive media coverage and public attention. The 2023 GPT-4 was praised for \nits increased accuracy and as a "holy grail" for its multimodal capabilities. OpenAI did not reveal the \nhigh-level architecture and the number of parameters of GPT-4. The release of ChatGPT led to an uptick \nin LLM usage across several research subfields of computer science, including robotics, software'}


In [31]:
#Prepare prompt for LLM
context=[]
#Append all returned chunks
for i in range(len(results[0])):
    context.append(results[0][1].entity.get("rag_text"))
#Create a prompt
prompt = ("Based on only the context provided, answer the query below: "
    +"Context:" + str(context)
    + "\n\n Query:"+ query)
print(prompt)

Based on only the context provided, answer the query below: Context:['research usage of BERT began to decline in 2023, following rapid improvements in the abilities of \ndecoder-only models (such as GPT) to solve tasks via prompting. \nAlthough decoder-only GPT-1 was introduced in 2018, it was GPT-2 in 2019 that caught widespread \nattention because OpenAI claimed to have initially deemed it too powerful to release publicly, out of \nfear of malicious use. GPT-3 in 2020 went a step further and as of 2025 is available only via API with no', 'research usage of BERT began to decline in 2023, following rapid improvements in the abilities of \ndecoder-only models (such as GPT) to solve tasks via prompting. \nAlthough decoder-only GPT-1 was introduced in 2018, it was GPT-2 in 2019 that caught widespread \nattention because OpenAI claimed to have initially deemed it too powerful to release publicly, out of \nfear of malicious use. GPT-3 in 2020 went a step further and as of 2025 is available 

In [35]:
#Generate with LLM
from langchain.llms import OpenAI
llm = OpenAI(temperature=0, model="gpt-3.5-turbo-instruct")
completion=llm(prompt)
print(completion)



GPT-1 is a decoder-only model introduced in 2018 that was overshadowed by the release of GPT-2 in 2019.
